# Modélisation — Régression Logistique

Classification multiclasse de la gravité des accidents avec compensation du déséquilibre (`class_weight='balanced'`).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

## 2. Chargement et Nettoyage des Données

In [ ]:
base_path = Path('..').resolve()
data_path = base_path / 'data' / 'raw' / 'accidents_2024.csv'

df = pd.read_csv(data_path, sep=';', encoding='latin-1')
df.drop_duplicates(inplace=True)
df.dropna(subset=['Gravité (label)'], inplace=True)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f"Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")

## 3. Encodage, Découpage Train/Test et Normalisation

In [ ]:
X_raw = df.drop(columns=['Num_Acc', 'Département', 'Gravité (label)'])
y = df['Gravité (label)']

X_encoded = pd.get_dummies(X_raw, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train : {X_train_scaled.shape} | Test : {X_test_scaled.shape}")

## 4. Entraînement et Évaluation

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.2%}\n")
print(classification_report(y_test, y_pred))

## 5. Validation Croisée (5-Fold Stratifié)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print(f"Scores par fold : {cv_scores.round(4)}")
print(f"Accuracy moyenne : {cv_scores.mean():.2%}")
print(f"Écart-type       : {cv_scores.std():.4f}")

## 6. Matrice de Confusion

In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title('Matrice de confusion — Régression Logistique', fontsize=12, fontweight='bold')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()

## 7. Coefficients par Variable (Classe « Tué »)

In [ ]:
coefficients = pd.DataFrame({
    'Variable': X_encoded.columns,
    'Coefficient': model.coef_[2]
})
print(coefficients.sort_values(by='Coefficient', ascending=False).head(10))

## 8. Facteurs Aggravants et Protecteurs

In [ ]:
index_tue = list(model.classes_).index('Tué')

coefficients_tue = pd.DataFrame({
    'Facteur (Variable)': X_encoded.columns,
    'Coefficient (Impact)': model.coef_[index_tue]
})

print("--- TOP 5 FACTEURS AGGRAVANTS (augmentent le risque de décès) ---")
print(coefficients_tue.sort_values(by='Coefficient (Impact)', ascending=False).head(5).to_string(index=False))

print("\n--- TOP 5 FACTEURS PROTECTEURS (diminuent le risque de décès) ---")
print(coefficients_tue.sort_values(by='Coefficient (Impact)', ascending=True).head(5).to_string(index=False))